# Enterprise Commodities & Energy Credit Risk & Capital Engine (ECCRE)
## 01: Exploratory Data Analysis & Volatility Feature Engineering

This notebook analyzes financial statement distributions and trailing 252-day commodity price volatility across wholesale energy counterparties.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_raw_data, load_commodity_prices
from src.features.ratios import compute_commodity_ratios, compute_trailing_volatility

sns.set_theme(style="darkgrid")
%matplotlib inline


### 1. Ingest Raw Counterparty Financials & Daily Commodity Prices


In [ ]:
df_financials = load_raw_data("data/raw/counterparty_financials.csv")
df_prices = load_commodity_prices("data/raw/commodity_prices.csv")
print(f"Loaded {len(df_financials)} counterparties.")
df_financials.head()


### 2. Commodity Log-Return Volatility Calculation (252-Day Trailing)


In [ ]:
price_cols = [c for c in df_prices.columns if c != "date"]
vol_summary = {}

for col in price_cols:
    vol = compute_trailing_volatility(df_prices[col], window=252)
    vol_summary[col] = vol

df_vol = pd.DataFrame(list(vol_summary.items()), columns=["Commodity", "Annualized Volatility (252d)"])
df_vol


### 3. Volatility-Adjusted Solvency & Coverage Ratios


In [ ]:
df_features = compute_commodity_ratios(df_financials)
df_features[["counterparty_name", "sector", "raw_icr", "commodity_vol_252d", "vol_adj_icr", "leverage_ratio", "liquidity_ratio"]].head(10)


### 4. Correlation Analysis: Leverage, ICR, Volatility, and Default Events


In [ ]:
plt.figure(figsize=(10, 6))
numeric_feats = ["vol_adj_icr", "leverage_ratio", "liquidity_ratio", "cash_margin_ratio", "commodity_vol_252d", "target"]
corr = df_features[numeric_feats].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Matrix: Engineered Ratios vs. Default Target")
plt.tight_layout()
plt.show()
